In [ ]:
!pip install --quiet openai
import os, json
from tqdm.auto import tqdm
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()


In [ ]:
ASPECT_LABELS = [
    "ambience",
    "food",
    "menu",
    "place",
    "price",
    "service",
    "staff",
    "miscellaneous",
]

POLARITY_LABELS = ["positive", "negative", "neutral"]

EMOTION_LABELS = [
    "Admiration",
    "Annoyance",
    "Appreciation",
    "Approval",
    "Confusion",
    "Disappointment",
    "Disapproval",
    "Disgust",
    "Excitement",
    "Fear",
    "Frustration",
    "Gratitude",
    "Impressed",
    "Indifferent",
    "Joy",
    "Realization",
    "Relaxation",
    "Relief",
    "Remorse",
    "Satisfaction",
    "Surprise",
]

ASPECT_POLARITY_ALLOWED_EMOTIONS = {
    "ambience": {
        "positive": ["Admiration", "Approval", "Relaxation", "Excitement", "Impressed"],
        "negative": ["Annoyance", "Disappointment", "Disapproval", "Confusion", "Fear"],
        "neutral": ["Indifferent"],
    },
    "food": {
        "positive": ["Joy", "Admiration", "Impressed", "Satisfaction"],
        "negative": ["Disgust", "Disappointment", "Annoyance", "Confusion", "Remorse"],
        "neutral": ["Indifferent"],
    },
    "menu": {
        "positive": ["Approval", "Excitement", "Appreciation", "Impressed"],
        "negative": ["Disappointment", "Annoyance", "Confusion"],
        "neutral": ["Indifferent"],
    },
    "place": {
        "positive": ["Approval", "Admiration", "Relief"],
        "negative": ["Fear", "Disappointment", "Annoyance", "Confusion"],
        "neutral": ["Indifferent"],
    },
    "price": {
        "positive": ["Approval", "Joy", "Satisfaction", "Surprise", "Relief"],
        "negative": ["Disappointment", "Disapproval", "Annoyance", "Frustration"],
        "neutral": ["Indifferent", "Realization"],
    },
    "service": {
        "positive": ["Approval", "Admiration", "Gratitude"],
        "negative": ["Annoyance", "Disappointment", "Frustration"],
        "neutral": ["Indifferent"],
    },
    "staff": {
        "positive": ["Gratitude", "Admiration", "Satisfaction"],
        "negative": ["Disappointment", "Frustration", "Annoyance"],
        "neutral": ["Indifferent"],
    },
    "miscellaneous": {
        "positive": ["Admiration", "Approval", "Joy", "Satisfaction", "Relief", "Surprise", "Excitement"],
        "negative": ["Annoyance", "Disappointment", "Disapproval", "Confusion", "Frustration", "Fear"],
        "neutral": ["Indifferent", "Realization"],
    },
}


In [ ]:
ABSA_SCHEMA = {
    "type": "object",
    "properties": {
        "predictions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "aspect":  {"type": "string", "enum": ASPECT_LABELS},
                    "polarity":{"type": "string", "enum": POLARITY_LABELS},
                    "emotion": {"type": "string", "enum": EMOTION_LABELS},
                },
                "required": ["aspect", "polarity", "emotion"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["predictions"],
    "additionalProperties": False,
}


In [ ]:
def classify_review_with_gpt(text: str):
    # Build constraints text for the prompt
    constraint_lines = []
    for aspect, pol_map in ASPECT_POLARITY_ALLOWED_EMOTIONS.items():
        for pol, emos in pol_map.items():
            constraint_lines.append(
                f"- If aspect = '{aspect}' and polarity = '{pol}', emotion MUST be one of: {', '.join(emos)}."
            )
    constraints_text = "\n".join(constraint_lines)

    system_msg = (
        "You are an expert annotator for aspect-based sentiment and emotion analysis "
        "in the restaurant domain.\n\n"
        "Given ONE restaurant review, extract all (aspect, polarity, emotion) triples.\n\n"
        "ASPECT labels: " + ", ".join(ASPECT_LABELS) + ".\n"
        "POLARITY labels: " + ", ".join(POLARITY_LABELS) + ".\n"
        "Global EMOTION labels: " + ", ".join(EMOTION_LABELS) + ".\n\n"
        "CONSTRAINTS:\n"
        f"{constraints_text}\n\n"
        "Other rules:\n"
        "1. Multi-aspect is allowed (0, 1 or many triples).\n"
        "2. Only annotate emotions clearly expressed or strongly implied; do NOT guess.\n"
        "3. If no aspect is evaluated, return an empty list.\n"
        "4. Output MUST follow the JSON schema exactly."
    )

    user_msg = f"Review: {text}"

    completion = client.chat.completions.create(
        model="gpt-4.1-mini",  # or "gpt-4.1"
        temperature=0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "absa_emotion_prediction",
                "schema": ABSA_SCHEMA,
                "strict": True,
            },
        },
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ],
    )

    data = json.loads(completion.choices[0].message.content)
    raw_preds = data.get("predictions", [])

    clean_preds = []
    for p in raw_preds:
        a = p.get("aspect")
        pol = p.get("polarity")
        emo = p.get("emotion")

        # basic checks
        if a not in ASPECT_LABELS or pol not in POLARITY_LABELS or emo not in EMOTION_LABELS:
            continue

        allowed = ASPECT_POLARITY_ALLOWED_EMOTIONS.get(a, {}).get(pol, [])

        if emo not in allowed:
            # try to repair: fallback to Indifferent if allowed, else drop
            if "Indifferent" in allowed:
                emo = "Indifferent"
            else:
                continue

        clean_preds.append({"aspect": a, "polarity": pol, "emotion": emo})

    return clean_preds


In [ ]:
test_text = "We were able to reserve a spot at the chef tasting bar with Morimoto who actually called in sick that night, but we were still charged full price."
print(classify_review_with_gpt(test_text))


[{'aspect': 'place', 'polarity': 'negative', 'emotion': 'Disappointment'}, {'aspect': 'price', 'polarity': 'negative', 'emotion': 'Disapproval'}]


In [ ]:
input_path  = "/content/cleaned_250.jsonl"        # adjust if needed
output_path = "/content/gpt_predictions.jsonl"

def make_gpt_predictions_file(input_path, output_path):
    count = 0
    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in tqdm(fin, desc="Annotating with GPT"):
            line = line.strip()
            if not line:
                continue

            item = json.loads(line)
            text = item.get("input")
            if not text:
                continue

            preds = classify_review_with_gpt(text)

            # IMPORTANT: this matches what 13_metrics expects:
            # input + output (predicted triples)
            out_item = {
                "input": text,
                "output": preds
            }

            json.dump(out_item, fout, ensure_ascii=False)
            fout.write("\n")
            count += 1

    print("Wrote", count, "lines to", output_path)

make_gpt_predictions_file(input_path, output_path)


Annotating with GPT: 0it [00:00, ?it/s]

Wrote 252 lines to /content/gpt_predictions.jsonl


In [ ]:
GOLD_PATH = "/content/cleaned_250.jsonl"        # our gold dataset
PRED_PATH = "/content/gpt_predictions.jsonl"    # generated by GPT

print("Gold:", GOLD_PATH)
print("Pred:", PRED_PATH)




Gold: /content/cleaned_250.jsonl
Pred: /content/gpt_predictions.jsonl


In [ ]:
!pip install -q scikit-learn

import json
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support, classification_report


In [ ]:
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

gold_data = load_jsonl(GOLD_PATH)
pred_data = load_jsonl(PRED_PATH)

print("Gold rows:", len(gold_data))
print("Pred rows:", len(pred_data))
assert len(gold_data) == len(pred_data), "Gold and pred must have same number of rows"


Gold rows: 252
Pred rows: 252


In [ ]:
def triples_by_aspect(triples):
    """
    triples: list of {"aspect": ..., "polarity": ..., "emotion": ...}
    returns: dict[aspect] = {"polarity": ..., "emotion": ...}
    Assumes at most one triple per aspect.
    """
    d = {}
    for t in triples:
        a = t.get("aspect")
        if not a:
            continue
        d[a] = {"polarity": t.get("polarity"), "emotion": t.get("emotion")}
    return d


In [ ]:
ASPECT_LABELS = [
    "ambience",
    "food",
    "menu",
    "place",
    "price",
    "service",
    "staff",
    "miscellaneous",
]

# collect aspect sets
y_true_aspects = []
y_pred_aspects = []

for g, p in zip(gold_data, pred_data):
    g_triples = g.get("output", [])
    p_triples = p.get("output", [])

    g_aspects = {t["aspect"] for t in g_triples if "aspect" in t}
    p_aspects = {t["aspect"] for t in p_triples if "aspect" in t}

    y_true_aspects.append(list(g_aspects))
    y_pred_aspects.append(list(p_aspects))

mlb = MultiLabelBinarizer(classes=ASPECT_LABELS)
mlb.fit([[]])  # just to initialise with given classes

Y_true = mlb.transform(y_true_aspects)
Y_pred = mlb.transform(y_pred_aspects)

prec_a, rec_a, f1_a, support_a = precision_recall_fscore_support(
    Y_true, Y_pred, average=None, labels=range(len(ASPECT_LABELS))
)

aspect_macro_f1 = f1_a.mean()

print("=== ASPECT CLASSIFICATION ===")
for label, f1, sup in zip(ASPECT_LABELS, f1_a, support_a):
    print(f"{label:12s} F1={f1:.4f}  support={sup}")
print(f"\nAspect Macro F1: {aspect_macro_f1:.4f}")


=== ASPECT CLASSIFICATION ===
ambience     F1=0.7671  support=39
food         F1=0.6745  support=164
menu         F1=0.8605  support=46
place        F1=0.4176  support=48
price        F1=0.7671  support=33
service      F1=0.5229  support=56
staff        F1=0.5695  support=104
miscellaneous F1=0.0702  support=46

Aspect Macro F1: 0.5812


In [ ]:
POLARITY_LABELS = ["positive", "negative", "neutral"]

# Collect polarity labels
y_true_pol = []
y_pred_pol = []

# Collect emotion labels
y_true_emo = []
y_pred_emo = []

for g, p in zip(gold_data, pred_data):
    g_triples = triples_by_aspect(g.get("output", []))
    p_triples = triples_by_aspect(p.get("output", []))

    common_aspects = set(g_triples.keys()) & set(p_triples.keys())

    for a in common_aspects:
        g_pol = g_triples[a].get("polarity")
        p_pol = p_triples[a].get("polarity")
        g_emo = g_triples[a].get("emotion")
        p_emo = p_triples[a].get("emotion")

        if g_pol is not None and p_pol is not None:
            y_true_pol.append(g_pol)
            y_pred_pol.append(p_pol)

        if g_emo is not None and p_emo is not None:
            y_true_emo.append(g_emo)
            y_pred_emo.append(p_emo)

print("Number of aspect matches used for polarity/emotion eval:", len(y_true_pol))


Number of aspect matches used for polarity/emotion eval: 283


In [ ]:
# Polarity metrics
print("\n=== POLARITY CLASSIFICATION ===")
pol_prec, pol_rec, pol_f1, pol_sup = precision_recall_fscore_support(
    y_true_pol, y_pred_pol, labels=POLARITY_LABELS, zero_division=0
)
pol_macro_f1 = pol_f1.mean()

for lab, f1, sup in zip(POLARITY_LABELS, pol_f1, pol_sup):
    print(f"{lab:8s} F1={f1:.4f}  support={sup}")
print(f"\nPolarity Macro F1: {pol_macro_f1:.4f}")


# Emotion metrics
EMOTION_LABELS = sorted({t["emotion"]
                         for g in gold_data
                         for t in g.get("output", [])
                         if "emotion" in t})

print("\n=== EMOTION CLASSIFICATION ===")
emo_prec, emo_rec, emo_f1, emo_sup = precision_recall_fscore_support(
    y_true_emo, y_pred_emo, labels=EMOTION_LABELS, zero_division=0
)

emotion_macro_f1 = emo_f1.mean()
emotion_micro_prec, emotion_micro_rec, emotion_micro_f1, _ = precision_recall_fscore_support(
    y_true_emo, y_pred_emo, average="micro", zero_division=0
)

for lab, f1, sup in zip(EMOTION_LABELS, emo_f1, emo_sup):
    print(f"{lab:14s} F1={f1:.4f}  support={sup}")

print(f"\nEmotion Macro F1: {emotion_macro_f1:.4f}")
print(f"Emotion Micro F1: {emotion_micro_f1:.4f}")



=== POLARITY CLASSIFICATION ===
positive F1=0.8778  support=106
negative F1=0.8218  support=123
neutral  F1=0.2571  support=54

Polarity Macro F1: 0.6523

=== EMOTION CLASSIFICATION ===
Admiration     F1=0.1667  support=32
Annoyance      F1=0.5505  support=43
Approval       F1=0.4490  support=25
Confusion      F1=0.4286  support=6
Disappointment F1=0.3762  support=38
Disapproval    F1=0.2727  support=18
Excitement     F1=0.0000  support=2
Fear           F1=0.0000  support=0
Frustration    F1=0.2857  support=15
Gratitude      F1=0.4615  support=4
Impressed      F1=0.2500  support=12
Indifferent    F1=0.2192  support=59
Joy            F1=0.3571  support=9
Relaxation     F1=0.6000  support=5
Relief         F1=0.0000  support=2
Satisfaction   F1=0.3243  support=12
Surprise       F1=0.0000  support=1

Emotion Macro F1: 0.2789
Emotion Micro F1: 0.3534


In [ ]:
def triples_set(triples):
    return {(t.get("aspect"), t.get("polarity"), t.get("emotion"))
            for t in triples if "aspect" in t and "polarity" in t and "emotion" in t}

n = len(gold_data)
correct_full = 0

for g, p in zip(gold_data, pred_data):
    gold_set = triples_set(g.get("output", []))
    pred_set = triples_set(p.get("output", []))
    if gold_set == pred_set:
        correct_full += 1

full_absa_accuracy = correct_full / n if n > 0 else 0.0

print("\n=== FULL ABSA TRIPLE MATCH ===")
print(f"Full ABSA Accuracy: {full_absa_accuracy:.4f}  ({correct_full}/{n} sentences)")



=== FULL ABSA TRIPLE MATCH ===
Full ABSA Accuracy: 0.0238  (6/252 sentences)
